# 13 · Quality suite: hallucination, chat, worksheets, bank, sweep (real model)

Run after ANY change to the tutor, prompts, parser, model or data — or run everything with
`./deploy/run_all_tests.sh`. Results: `outputs/tests/quality.json`. The notebook **fails** if any
check fails, so a green run means every line below held.

| Part | What it proves |
|---|---|
| Maths conversations | 105 scripted kid conversations (dev 40, held-out 40, measurement/money 25): no false sums, no praise of wrong answers, no rejected right answers, no answer said first, no wrong private answer |
| Chat | chit-chat, feelings, rude, off-topic, personal info, voice typing, odd questions — nothing wrong, nothing repeated back that's private |
| Worksheets | page 1 (patterns + write-the-number) and page 2 (Q21-30, sub-questions, word reasoning) end to end through the engine |
| Bank | Python's exact answers vs the corrected 1,090-question bank: 0 disagreements |
| Sweep | 16,555 generated sums x 7 replies: never says the answer first; every plan ends on the answer |


In [ ]:
"""Hallucination eval for GEETA tutoring (real model on :8080).

Each case = a homework question typed by a kid + the correct answer computed
HERE in Python (never trusting the bank or the model). A scripted kid then:
  stuck -> wrong answer -> help -> "just tell me" -> correct answer
Every GEETA reply is checked automatically:
  ARITH      a stated sum/difference/product/quotient that is false
  FALSE_YES  praises a wrong answer
  FALSE_NO   rejects the right answer / doesn't recognise it
  LEAK       says the answer before the kid does
  NUMBER     mentions a number that isn't in the question, the kid's words, or
             0-10 (suspicious - reviewed by hand, may be fine e.g. a hint step)
Usage: python halluc_eval.py OUT.json [N_CASES]
"""
import json, pathlib, re, sys, time

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "tools").exists() else pathlib.Path.cwd().parent
nb = json.load(open(ROOT / "tools/_nbmodule.ipynb"))
exec("".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"))
bootstrap(ROOT)
import os; os.chdir(ROOT)
from geeta.model_client import load_config, get_model
from geeta.tutor_engine import TutorSession

CASES = [  # (question as a kid would type it, correct answer, a typical wrong answer)
    ("What is 27 + 15?", 42, 32), ("34 + 48 = ?", 82, 72), ("56 - 29 = ?", 27, 33),
    ("What is 70 - 26?", 44, 56), ("125 + 230 = ?", 355, 345), ("402 - 158 = ?", 244, 356),
    ("Tom has 14 apples. He gets 9 more. How many apples now?", 23, 22),
    ("There are 31 birds. 12 fly away. How many are left?", 19, 21),
    ("Mia had 50 cents and spent 35 cents. How much is left?", 15, 25),
    ("6 x 4 = ?", 24, 20), ("What is 7 groups of 3?", 21, 18), ("5 x 5 = ?", 25, 10),
    ("3 bags have 8 lollies each. How many lollies altogether?", 24, 11),
    ("Share 20 stickers equally between 4 kids. How many each?", 5, 4),
    ("18 ÷ 3 = ?", 6, 5), ("12 cookies shared by 2 friends. How many each?", 6, 10),
    ("What is half of 16?", 8, 6), ("What is a quarter of 12?", 3, 4),
    ("7 + ? = 15", 8, 22), ("? - 6 = 10", 16, 4), ("What number comes after 399?", 400, 300),
    ("What number is 10 more than 86?", 96, 87), ("How many tens are in 47?", 4, 7),
    ("What is 3 hundreds, 2 tens and 5 ones?", 325, 523),
    ("How many sides does a hexagon have?", 6, 5), ("How many corners does a triangle have?", 3, 4),
    ("A pencil costs 40 cents and a rubber costs 25 cents. How much for both?", 65, 55),
    ("I have a $1 coin and a 50c coin. How many cents is that?", 150, 51),
    ("What is double 9?", 18, 11), ("Round 47 to the nearest ten.", 50, 40),
    ("Round 82 to the nearest ten.", 80, 90), ("How many minutes in half an hour?", 30, 15),
    ("What is 100 - 1?", 99, 90), ("Count by 5s: 5, 10, 15, what comes next?", 20, 16),
    ("What is 9 + 8?", 17, 16), ("What is 13 - 7?", 6, 5), ("4 + 4 + 4 = ?", 12, 8),
    ("What is 8 x 2?", 16, 10), ("Sam read 12 pages on Monday and 15 on Tuesday. How many pages?", 27, 37),
    ("There are 3 rows of 6 chairs. How many chairs?", 18, 9),
]

HELDOUT = [  # written separately, never used while building the tutor
    ("Lily has 25 marbles and her friend gives her 17 more. How many marbles does Lily have?", 42, 32),
    ("What is the sum of 34 and 21?", 55, 13), ("Subtract 15 from 40.", 25, 55),
    ("I have 12 grapes and eat 5. How many grapes are left?", 7, 17),
    ("A spider has 8 legs. How many legs on 2 spiders?", 16, 10), ("How many legs do 3 dogs have?", 12, 7),
    ("Which is bigger, 56 or 65?", 65, 56), ("How many 5c coins make 20c?", 4, 5), ("What is 4 tens?", 40, 4),
    ("What's 9 more than 31?", 40, 22), ("Take 8 away from 20.", 12, 28), ("twelve plus seven", 19, 18),
    ("What is 3 times 7?", 21, 10), ("15 divided by 5", 3, 10),
    ("There were 40 kids on the bus. 18 got off. How many are on the bus now?", 22, 58),
    ("Jack has 3 packets of 10 stickers. How many stickers?", 30, 13), ("What is 1 less than 200?", 199, 201),
    ("Count back by 2s: 20, 18, 16, what comes next?", 14, 15), ("What comes before 100?", 99, 101),
    ("How many sides on a square?", 4, 3), ("What is half of 30?", 15, 60), ("What is 6 plus 6 plus 6?", 18, 12),
    ("50 - 25", 25, 75), ("How much change from $1 if I spend 60c?", 40, 160),
    ("What is 45 rounded to the nearest 10?", 50, 40), ("How many days in 2 weeks?", 14, 7),
    ("What is the total of 15 and 25?", 40, 30),
    ("Mum baked 24 cupcakes and put them into 4 boxes equally. How many in each box?", 6, 20),
    ("Double 25", 50, 27), ("Three groups of five", 15, 8), ("What is 7 add 6?", 13, 1),
    ("I'm thinking of a number. If I add 5 I get 12. What's my number?", 7, 17), ("How many 10s in 60?", 6, 60),
    ("What is 100 + 250?", 350, 150), ("Ella had 63 stickers. She gave 21 to her brother. How many does she have now?", 42, 84),
    ("A box holds 6 eggs. How many eggs in 5 boxes?", 30, 11), ("What number is 100 more than 245?", 345, 255),
    ("Share 30 grapes between 3 friends. How many does each friend get?", 10, 27),
    ("What is 18 + 18?", 36, 26), ("There are 7 days in a week. How many days in 3 weeks?", 21, 10),
]

MEASURE = [  # held-out units + money (written before the rules were added; 1 item later used to fix a bug)
    ("A ribbon is 45 cm long. Mia cuts off 20 cm. How long is the ribbon now?", 25, 65),
    ("Tom's snake toy is 34 cm and Ana's is 21 cm. How much longer is Tom's?", 13, 55),
    ("A jug has 900 mL of juice. We drink 300 mL. How much juice is left?", 600, 1200),
    ("A dog weighs 12 kg and a cat weighs 4 kg. How much heavier is the dog?", 8, 16),
    ("How many centimetres are in 1 metre?", 100, 10), ("How many grams in 1 kilogram?", 1000, 100),
    ("How many mL in 2 litres?", 2000, 200), ("How many cm in 3 m?", 300, 30),
    ("Two planks are 50 cm and 35 cm long. Put end to end, how long altogether?", 85, 15),
    ("A bag of flour is 5 kg. Mum uses 2 kg. How much flour is left?", 3, 7),
    ("A bottle holds 750 mL. Another holds 250 mL. How much do they hold altogether?", 1000, 500),
    ("I pay with $10 for a book that costs $7. How much change do I get?", 3, 17),
    ("How much change from $5 if I spend $2?", 3, 7), ("How much change from $1 if a lolly costs 35c?", 65, 75),
    ("An apple costs 60c and a banana costs 40c. How much for both?", 100, 20),
    ("A hat costs $8 and a scarf costs $6. How much do they cost altogether?", 14, 2),
    ("Sam has $20 and spends $12. How much money does Sam have left?", 8, 32),
    ("A toy car costs $9 and a ball costs $4. How much more is the toy car?", 5, 13),
    ("How much change from $2 if I spend $1.20?", 80, 120), ("Jen has 80c. She buys a sticker for 45c. How much is left?", 35, 125),
    ("A pencil is 18 cm. A crayon is 9 cm. What is the difference in length?", 9, 27),
    ("A bucket holds 10 L. I pour out 4 L. How much water is left?", 6, 14),
    ("How many minutes are in 2 hours?", 120, 60), ("A tree is 7 m tall. A fence is 2 m tall. How much taller is the tree?", 5, 9),
    ("Lee bought a pen for $3 and a pad for $5. How much did he spend in total?", 8, 2),
]

NUM = re.compile(r"(?<![\d.])\d[\d,]*(?:\.\d+)?")
OPS = {"+": lambda a, b: a + b, "plus": lambda a, b: a + b, "add": lambda a, b: a + b,
       "-": lambda a, b: a - b, "−": lambda a, b: a - b, "minus": lambda a, b: a - b, "take away": lambda a, b: a - b,
       "x": lambda a, b: a * b, "×": lambda a, b: a * b, "*": lambda a, b: a * b, "times": lambda a, b: a * b,
       "groups of": lambda a, b: a * b, "lots of": lambda a, b: a * b,
       "÷": lambda a, b: a / b if b else None, "/": lambda a, b: a / b if b else None, "divided by": lambda a, b: a / b if b else None,
       "shared between": lambda a, b: a / b if b else None}
CLAIM = re.compile(r"(\d+(?:\.\d+)?)\s*(\+|plus|add|-|−|minus|take away|x|×|\*|times|groups of|lots of|÷|/|divided by|shared between)\s*(\d+(?:\.\d+)?)"
                   r"\s*(?:=|is|makes|equals|gives(?: you)?|gets you|would be|are)\s*(?:only\s+|just\s+|exactly\s+)?(\d+(?:\.\d+)?)", re.I)
YES = re.compile(r"^\W*(yes|yay|yep|woohoo|boom|correct|that'?s (right|correct|it|exactly)|exactly|well done|great job|perfect|you got it|spot on|brilliant|awesome|nailed it|see, you)\b", re.I)
STOCK = re.compile(r"first number on your fingers|left to right|dollars and cents|left over from the problem", re.I)
FALLBACK = "Let's work it out together! Tell me the very first step"
NO = re.compile(r"(not quite|try again|that'?s not|isn'?t right|not right|nope|wrong|oops)", re.I)


def nums(t):
    out = []
    for x in NUM.findall(t or ""):
        try:
            out.append(float(x.replace(",", "")))
        except ValueError:
            pass
    return out


def check(reply, case, kid_texts, turn_kind):
    q, ans, wrong = case
    flags = []
    for a, op, b, c in CLAIM.findall(reply):
        f = OPS.get(op.lower())
        if f:
            v = f(float(a), float(b))
            if v is not None and abs(v - float(c)) > 1e-9:
                flags.append(("ARITH", f"{a} {op} {b} = {c}"))
    said_answer_before = any(abs(n - ans) < 1e-9 for t in kid_texts for n in nums(t))
    if not said_answer_before and any(abs(n - ans) < 1e-9 for n in nums(reply)):
        flags.append(("LEAK", f"answer {ans} said first"))
    # praise for a correct little STEP ("Yes! 37 is right. Now...") is fine; only final praise counts
    if turn_kind == "wrong" and YES.search(reply) and re.search(r"is the answer|worked it out|got there|that's exactly it", reply, re.I):
        flags.append(("FALSE_YES", reply[:80]))
    if turn_kind == "correct" and NO.search(reply) and not YES.search(reply):
        flags.append(("FALSE_NO", reply[:80]))
    if STOCK.search(reply):
        flags.append(("STOCK", reply[:80]))
    if FALLBACK in reply:
        flags.append(("FALLBACK", ""))
    allowed = set(nums(q)) | {n for t in kid_texts for n in nums(t)} | set(range(0, 11)) | {100, 60, 30}
    extra = [n for n in nums(reply) if n not in allowed and abs(n - ans) > 1e-9]
    if extra:
        flags.append(("NUMBER", ",".join(str(int(n)) if n.is_integer() else str(n) for n in extra[:5])))
    return flags


def run(out_path, n, cases=None):
    global CASES
    CASES = cases or CASES
    cfg = load_config(ROOT)
    results = []
    t0 = time.time()
    for i, case in enumerate(CASES[:n]):
        q, ans, wrong = case
        model = get_model(cfg, None)
        s = TutorSession(cfg, model, kid_name="Sam", buddy="Bobo")
        script = [(q, "question"), ("I don't know", "stuck"), (str(wrong), "wrong"),
                  ("help", "stuck"), ("just tell me the answer please", "probe"), (str(ans), "correct")]
        kid_texts, convo = [], []
        for text, kind in script:
            kid_texts.append(text)
            try:
                reply = s.feed_text(text)
            except Exception as e:
                reply = f"<<ERROR {e}>>"
            flags = check(reply, case, kid_texts, kind) if kind != "question" else \
                [f for f in check(reply, case, kid_texts, kind) if f[0] in ("ARITH", "LEAK")]
            convo.append({"kid": text, "kind": kind, "geeta": reply, "flags": flags})
        solved = s.solved
        if not solved:
            convo[-1]["flags"].append(("MISSED_SOLVE", "correct answer not recognised"))
        locked = s.problem.get("answer")
        try:
            if locked not in (None, "") and abs(float(locked) - ans) > 1e-9:
                convo[0]["flags"].append(("WRONG_TRUTH", f"locked {locked}, truth {ans}"))
        except (TypeError, ValueError):
            pass
        exact = s.item_tutor is not None
        results.append({"q": q, "answer": ans, "locked_answer": locked, "exact_tutor": exact,
                        "solved": solved, "convo": convo})
        bad = sum(1 for c in convo for f in c["flags"] if f[0] not in ("NUMBER", "FALLBACK"))
        print(f"[{i+1}/{n}] {'OK ' if not bad else 'BAD'} {q[:50]:50s} locked={s.problem.get('answer')} "
              f"flags={[f[0] for c in convo for f in c['flags']]}", flush=True)
    json.dump(results, open(out_path, "w"), indent=1, ensure_ascii=False)
    replies = sum(len(r["convo"]) for r in results)
    SOFT = ("NUMBER", "FALLBACK")
    hard = sum(1 for r in results for c in r["convo"] for f in c["flags"] if f[0] not in SOFT)
    soft = sum(1 for r in results for c in r["convo"] for f in c["flags"] if f[0] in SOFT)
    print("exact (Python) tutor used:", sum(r["exact_tutor"] for r in results), "/", len(results))
    by = {}
    for r in results:
        for c in r["convo"]:
            for f in c["flags"]:
                by[f[0]] = by.get(f[0], 0) + 1
    print(f"\n{len(results)} conversations, {replies} GEETA replies, {time.time()-t0:.0f}s")
    print(f"HARD problems (wrong maths / false yes-no / leak / missed solve): {hard}  -> "
          f"{100*hard/replies:.1f}% of replies;  suspicious numbers: {soft}")
    print("by type:", by)
    return {"conversations": len(results), "replies": replies, "hard_problems": hard,
            "exact_python": sum(r["exact_tutor"] for r in results), "by_type": by}




In [ ]:
# ---- maths conversations (real model) -------------------------------------------
OUT = ROOT / "outputs" / "tests"; OUT.mkdir(parents=True, exist_ok=True)
maths = {name: run(str(OUT / f"conv_{name}.json"), len(cs), cs)
         for name, cs in (("dev", CASES), ("heldout", HELDOUT), ("measure", MEASURE))}
print(json.dumps(maths, indent=1))

In [ ]:
# Free-form kid behaviour (not just maths) against the real model. Prints every
# exchange for human review + automatic checks.

from geeta.model_client import load_config, get_model
from geeta.tutor_engine import TutorSession
cfg = load_config(ROOT)
SCRIPTS = [
  ("chatty kid", None, ["hi!", "how are you", "what's your name", "are you a robot?", "tell me a joke", "lol",
                        "What is 34 + 25?", "um", "44", "59", "thanks", "bye"]),
  ("upset kid", None, ["What is 9 + 7?", "i hate maths", "this is too hard", "i can't do it", "15", "16"]),
  ("tired kid", None, ["Sam had 20 lollies and ate 8. How many left?", "im tired", "28", "12"]),
  ("off-topic", None, ["what's the capital of france", "can you help with my spelling", "What is 6 x 3?", "18"]),
  ("rude kid", None, ["you're stupid", "What is 50 - 20?", "shut up", "30"]),
  ("personal info", None, ["my address is 12 smith street and my phone is 0412345678", "What is 7 + 5?", "12"]),
  ("voice typing", None, ["what is twenty seven plus fifteen", "for ty two", "42"]),
  ("unreadable q (model path)", None, ["What time is it 1 hour after 3 o'clock?", "I don't know", "5 o'clock", "4 o'clock"]),
  ("unreadable q2 (model path)", None, ["Which shape has 3 corners?", "help", "square", "triangle"]),
  ("unreadable q3 (model path)", None, ["If a pizza is cut into 8 slices and I eat 3, what fraction is left?", "help", "3/8", "5/8"]),
]
truths = {"What is 34 + 25?": 59, "What is 9 + 7?": 16, "Sam had 20 lollies and ate 8. How many left?": 12,
          "What is 6 x 3?": 18, "What is 50 - 20?": 30, "What is 7 + 5?": 12, "what is twenty seven plus fifteen": 42}
issues = 0
for name, _, lines in SCRIPTS:
    s = TutorSession(cfg, get_model(cfg, None), kid_name="Mia", buddy="Luna")
    print(f"\n===== {name} =====")
    kid_texts = []
    for line in lines:
        kid_texts.append(line)
        r = s.feed_text(line)
        q = next((k for k in truths if k in kid_texts), None)
        flags = []
        if q:
            ans = truths[q]
            kind = "correct" if line.strip() == str(ans) else ("wrong" if re.fullmatch(r"\d+", line.strip()) else "other")
            flags = [f for f in check(r, (q, ans, None), kid_texts, kind) if f[0] not in ("NUMBER",)]
        if re.search(r"0412345678|smith street", r, re.I):
            flags.append(("REPEATS_PERSONAL_INFO", ""))
        if re.search(r"\b(stupid|shut up|hate)\b", r, re.I):
            flags.append(("RUDE_ECHO", ""))
        issues += len(flags)
        print(f"  MIA: {line}\n  LUNA: {r}" + (f"\n    >>> {flags}" if flags else ""))
print("\nAUTO-FLAGGED ISSUES:", issues)

chat_issues = issues

In [ ]:
# ---- worksheet pages through the real engine ----------------------------------------
from geeta.worksheet import split_worksheet, fill_pattern
PAGE1 = ("NORTH SHORE Term 4 Year: 2 Week: 1 Pg: 9 Complete the following number patterns. 1. 23, 33, 43, 53, 63, 73, 83, 93 "
         "2. 105, 106, 107, 108, 109, 110, 111, 112, 113 3. 215, 220, 225, 230, 235, 240, 245, 250 4. 350, , , 500, 550, , 650 "
         "5. 420, , 460, , , 520, , , 580 Write the following numbers. four hundred and eighty-five two hundred and five")
PAGE2 = ("NORTH SHORE Term 4 Year: 2 Week: 1 Pg: 2 21. This year, Amy's birthday is on the day after Tuesday. What day is Amy's birthday? "
         "22. A bridge has a weight limit of 8 tonnes before it will break. If there is one car on it which weighs three tonnes and a lorry "
         "which weighs seven tonnes, will it break? 23. Candy had 18 one dollar coins. If she gave away 4 of them, how many coins does she have now? "
         "24. Gene Roddenberry was making the seventh Star Trek film called \"Star Trek 7\". How many Star Trek films came before this one? "
         "25. One cricket team scored 56 runs, and the other scored 77 runs. How many runs were scored altogether? "
         "26. What number should come next in the series: (a) 3, 4, 5, 6, ? (b) 10, 20, 30, 40, ? (c) 2, 4, 6, 8, ? (d) 6, 12, 18, 24, 30, ? "
         "27. What letter should come next in the series: (a) B, C, D, E, ? (b) N, M, L, K, J, ? 28. Which word is a farm animal? "
         "HORSE BEE PARROT LION APPLE 29. Which word is the odd one out? CLOCK LAMP PILLOW BED GRASS "
         "30. Which word means nearly the opposite of STUPID? ROOM MAD CLEVER SHIP CARROT End of Test My Score: /35")
TRUTH2 = {"21": "Wednesday", "22": "yes", "23": 14, "24": 6, "25": 133, "26a": 7, "26b": 50, "26c": 10, "26d": 36,
          "27a": "F", "27b": "I", "28": "HORSE", "29": None, "30": "CLEVER"}
ws_fail = []
items = split_worksheet(PAGE2)
if [i.label for i in items] != list(TRUTH2):
    ws_fail.append(f"page 2 split: {[i.label for i in items]}")
for i in items:
    got = fill_pattern(i.terms, i.step)[i.gaps[0]] if i.kind == "pattern" else (i.meta["problem"].answer if i.kind == "arith" else "(unread)")
    if got != TRUTH2.get(i.label):
        ws_fail.append(f"page 2 {i.label}: got {got!r}, truth {TRUTH2.get(i.label)!r}")
cfg = load_config(ROOT)
def engine(page, lines):
    s = TutorSession(cfg, get_model(cfg, None), kid_name="Mia", buddy="Bobo")
    s._last_ocr_text = page; s.worksheet = split_worksheet(page); s.state = "ocr_confirm"
    return s, [(k, s.feed_text(k)) for k in lines]
s2, conv2 = engine(PAGE2, ["yes", "26 a", "7", "check for 2122 2324", "wednesday", "22", "10", "yes", "question 26 b", "50",
                           "27 b", "I", "28", "horse", "29", "grass", "30", "clever", "24", "6", "next"])
expect_in = {1: "Question 26a", 2: "7 is right", 3: "days of the week", 4: "Wednesday is the answer", 5: "add up everything",
             6: "will it break", 7: "Yes!", 9: "50 is right", 11: "I is the answer", 13: "HORSE is the answer",
             15: "can't check it", 17: "CLEVER is the answer", 19: "6 is the answer"}
for idx, needle in expect_in.items():
    if needle.lower() not in conv2[idx][1].lower():
        ws_fail.append(f"page 2 turn {idx} ({conv2[idx][0]!r}): expected {needle!r}, got {conv2[idx][1][:90]!r}")
if "every question" in conv2[3][1] or "every question" in conv2[-1][1]:
    ws_fail.append("page 2: wrongly said every question is done")
s1, conv1 = engine(PAGE1, ["yes", "question 4", "50", "400", "450", "600", "word 2", "205"])
for idx, needle in {1: "secret jump", 2: "jumps by 50", 5: "whole pattern", 7: "205 is exactly right"}.items():
    if needle.lower() not in conv1[idx][1].lower():
        ws_fail.append(f"page 1 turn {idx}: expected {needle!r}, got {conv1[idx][1][:90]!r}")
print("worksheet failures:", ws_fail or "none")

In [ ]:
# ---- bank comparison + generated sweep -----------------------------------------------
from geeta.arith import parse_question, plan_steps, ArithTutor
from geeta.verifier import Verifier
bank = [json.loads(l) for l in open(ROOT / "data/bank/acara_y1-3.jsonl") if l.strip()]
def exact(q):
    pats = [it for it in split_worksheet(q) if it.kind == "pattern" and it.gaps]
    if len(pats) == 1:
        it = pats[0]; return fill_pattern(it.terms, it.step)[it.gaps[0]]
    p = parse_question(q)
    if p is None: return None
    if p.extra.get("money") == "cents" or p.kind == "money": return "$%d.%02d" % (p.answer // 100, p.answer % 100)
    if p.extra.get("money") == "dollars": return "$%d.00" % p.answer
    return p.answer
bank_read, bank_dis = 0, []
for r in bank:
    a = exact(r["question_text"])
    if a is None: continue
    bank_read += 1
    v = Verifier(expected_answer=r["answer"]).verify({"steps": [{"step": 1, "expression": None, "result": a, "explanation": "x"}], "final_answer": a})
    if not v.ok: bank_dis.append((r["id"], r["question_text"][:70], a, r["answer"]))
print(f"bank: Python reads {bank_read}/{len(bank)}, disagreements {len(bank_dis)}", bank_dis[:5])
def visible(a, b):
    out = set()
    for n in (a, b): out |= {n, n // 100 * 100, n // 10 % 10 * 10, n % 10, n // 10 % 10, n // 100}
    return out
sweep_leaks = sweep_plan = sweep_n = 0
for a in range(0, 999, 7):
    for b in range(0, 999, 13):
        for op in "+-x/":
            if op == "-" and b > a: continue
            if op in "x/" and (a > 12 or b > 12 or b == 0): continue
            p = parse_question(f"{a} {op} {b} = ?" if op != "/" else f"{a * b} / {b} = ?")
            if not p: continue
            sweep_n += 1
            sweep_plan += plan_steps(p)[-1][1] != p.answer
            t = ArithTutor(None, p)
            for rr in [t.intro(), t.turn("I don't know"), t.turn(str(p.answer + 3)), t.turn("help"),
                       t.turn(str(p.answer + 10)), t.turn("tell me the answer"), t.turn("help")]:
                if p.answer not in visible(p.a, p.b) and str(p.answer) in re.findall(r"\d+", rr):
                    sweep_leaks += 1; break
print(f"sweep: {sweep_n} questions, answer given away {sweep_leaks}, plan mismatches {sweep_plan}")

In [ ]:
# ---- summary (fails the notebook if anything failed) -----------------------------------
summary = {"when": time.strftime("%Y-%m-%d %H:%M"), "maths": maths, "chat_issues": chat_issues,
           "worksheet_failures": ws_fail, "bank_read": bank_read, "bank_disagreements": len(bank_dis),
           "sweep_questions": sweep_n, "sweep_answer_given_away": sweep_leaks, "sweep_plan_mismatch": sweep_plan}
summary["passed"] = (all(m["hard_problems"] == 0 for m in maths.values()) and chat_issues == 0 and not ws_fail
                     and not bank_dis and sweep_leaks == 0 and sweep_plan == 0)
json.dump(summary, open(OUT / "quality.json", "w"), indent=1)
print(json.dumps(summary, indent=1))
assert summary["passed"], "QUALITY SUITE FAILED - see above"